# LLM 응답 캐시


> 업데이트 기준: **2026-09-18**  
> 책의 학습 목표는 유지하면서 LangChain 1.x의 분리된 provider 패키지와 현재 메시지/스트리밍 API에 맞췄습니다. 모델 이름은 공급자 정책에 따라 바뀔 수 있으므로 환경 변수로 덮어쓸 수 있게 구성했습니다.


LangChain 캐시는 동일한 직렬화된 입력과 모델 설정에 대한 결과를 재사용합니다. 개발·테스트에서 비용과 지연을 줄일 수 있지만, 최신성이 필요한 질문에는 적합하지 않습니다. 현재 캐시는 일반 `invoke` 호출에 사용하며 스트리밍 캐시로 간주하면 안 됩니다.


In [ ]:
%pip install -qU langchain-core langchain-openai langchain-community python-dotenv


In [ ]:
import os
import time
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    temperature=0,
)
prompt = PromptTemplate.from_template("{country}에 대해 200자 내외로 요약해 주세요.")
chain = prompt | llm | StrOutputParser()


def timed_invoke(country: str) -> str:
    started = time.perf_counter()
    result = chain.invoke({"country": country})
    print(f"elapsed={time.perf_counter() - started:.3f}s")
    return result


## 캐시를 끈 기준 호출


In [ ]:
from langchain_core.globals import set_llm_cache

set_llm_cache(None)
print(timed_invoke("한국"))


## 메모리 캐시

`InMemoryCache`는 현재 Python 프로세스가 종료되면 사라집니다.


In [ ]:
from langchain_core.caches import InMemoryCache

set_llm_cache(InMemoryCache())
print(timed_invoke("한국"))  # 첫 호출: 저장
print(timed_invoke("한국"))  # 두 번째 호출: 캐시 적중


## SQLite 캐시

프로세스를 다시 시작해도 남는 로컬 캐시입니다. 운영 환경에서는 동시성·보존 정책·민감정보 저장 여부를 별도로 설계해야 합니다.


In [ ]:
from pathlib import Path
from langchain_community.cache import SQLiteCache

cache_dir = Path("cache")
cache_dir.mkdir(parents=True, exist_ok=True)
set_llm_cache(SQLiteCache(database_path=str(cache_dir / "llm_cache.db")))

print(timed_invoke("한국"))
print(timed_invoke("한국"))


## 개별 모델에서 캐시 끄기

전역 캐시를 유지하면서 특정 모델 호출만 제외하려면 모델 생성 시 `cache=False`를 지정합니다.


In [ ]:
uncached_llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    temperature=0,
    cache=False,
)
print(uncached_llm.invoke("대한민국의 수도는 어디인가요?").text)
